<a href="https://colab.research.google.com/github/fraaaa25/Tugas-Proyek-EDA/blob/main/TugasProyekEDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama Anggota:
1.   Afra Haura Adzkiya Rahma / 01
2.   Shafa Almirah Zulkarnain / 35

Dataset yang Di Pilih : **Dataset nilai akademik siswa**

**Pertanyaan Analisis Awal:**
1. Berapa jumlah data siswa dalam dataset?
2. Mata pelajaran yang memiliki rata-rata nilai paling tinggi dan paling rendah?
3. Berapa jumlah data siswa yang memiliki data kosong (missing value)?

**Dugaan Masalah Kualitas Data**
*   Missing Value pada Kolom : Nilai, Guru Pengampu
*   Duplikat data
*   Tipe data tidak sesuai pada kolom : Nilai, Tanggal

**Rencana Teknik Pembersihan**

* Menangani missing value
Mengecek data kosong pada setiap kolom.
  - Nilai yang kosong akan diisi dengan rata-rata nilai.
  - Kolom guru_pengampu yang kosong akan diisi dengan keterangan “Tidak diketahui”.
* Menangani data duplikat
Mengecek jumlah data yang duplikat.
Menghapus baris yang duplikat menggunakan drop_duplicated() agar tidak terjadi penghitungan data secara berulang.
* Memperbaiki tipe data kolom nilai
Mengubah kolom nilai menjadi tipe bulat agar dapat digunakan untuk menghitung rata-rata nilai.
Memperbaiki format nilai yang tidak seragam seperti 73,0.
* Merapikan format tanggal
Menyamakan format pada kolom tanggal_ujian agar seluruh tanggal memiliki format yang konsisten.

**Rencana Manipulasi Data**
- Filter: Menampilkan data yang memiliki missing value.
- Sort: Mengurutkan nilai untuk melihat nilai tertinggi dan terendah.
- Kolom turunan: Tidak digunakan.
- Groupby/agregasi: Mengelompokkan berdasarkan mata_pelajaran untuk menghitung rata-rata nilai tiap mata pelajaran.






In [28]:
# Import Library
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv('dataset_nilai_akademik_siswa.csv')

In [30]:
# Menyimpan data asli sebelum cleaning
df_awal = df.copy()

Data Inspection

In [31]:
# Data Inspection
df.head()

,id_siswa,nama,kelas,mata_pelajaran,jenis_ujian,tanggal_ujian,nilai,guru_pengampu
0,SIS0004,Joko Prasetyo,XI RPL 2,KKA,uas,06/08/2026,46,Ibu Wati
1,SIS0020,Eka Putri,XI RPL 1,PKK,uts,12 Agustus 2026,"73,0",Bpk. Santoso
2,SIS0015,Nanda Pratama,XI RPL 3,Pemrograman Web,UAS,2026-08-15,69,Bpk. Santoso
3,SIS0046,Fajar Nugroho,XI RPL 2,Matematika,UH,10/08/2026,73,Bpk. Arifin
4,SIS0011,Ayu Lestari,XI RPL 2,Pemrograman Web,UH,6 Agustus 2026,53,Ibu Wati


In [32]:
print("Jumlah baris :", df.shape[0])
print("Jumlah kolom:", df.shape[1])

Jumlah baris : 79
Jumlah kolom: 8


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_siswa        79 non-null     object
 1   nama            79 non-null     object
 2   kelas           79 non-null     object
 3   mata_pelajaran  79 non-null     object
 4   jenis_ujian     79 non-null     object
 5   tanggal_ujian   79 non-null     object
 6   nilai           75 non-null     object
 7   guru_pengampu   75 non-null     object
dtypes: object(8)
memory usage: 5.1+ KB


In [34]:
df.describe(include='all')

,id_siswa,nama,kelas,mata_pelajaran,jenis_ujian,tanggal_ujian,nilai,guru_pengampu
count,79,79,79,79,79,79,75,75
unique,75,20,3,6,6,38,52,5
top,SIS0046,Lukman Hakim,XI RPL 3,PKK,UH,2026-08-05,73,Bpk. Arifin
freq,2,7,34,16,21,8,4,18


Data Cleaning

In [35]:
# Data Cleaning
print("Missing value setiap kolom:")
print(df.isnull().sum())

Missing value setiap kolom:
id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             4
guru_pengampu     4
dtype: int64


In [36]:
print("Jumlah data duplikat:", df.duplicated().sum())

Jumlah data duplikat: 4


In [37]:
df = df.drop_duplicates().reset_index(drop=True)
print("Jumlah data setelah menghapus duplikat:", len(df))

Jumlah data setelah menghapus duplikat: 75


In [38]:
# Mengubah koma menjadi titik
df['nilai'] = df['nilai'].astype('string').str.replace(
    ',', '.', regex=False
)
# Menghapus tulisan "poin"
df['nilai'] = df['nilai'].str.replace(
    ' poin', '', regex=False
)
# Mengubah nilai menjadi tipe numerik
df['nilai'] = pd.to_numeric(
    df['nilai'],
    errors='coerce'
)

In [39]:
df.loc[
    (df['nilai'] < 0) | (df['nilai'] > 100),
    'nilai'
] = np.nan

In [40]:
rata_rata_nilai = df['nilai'].mean()
df['nilai'] = df['nilai'].fillna(rata_rata_nilai)

In [41]:
df['guru_pengampu'] = df['guru_pengampu'].fillna(
    'Tidak diketahui'
)

In [42]:
# Mengubah nama bulan Indonesia menjadi angka
df['tanggal_ujian'] = df['tanggal_ujian'].str.replace(
    'Agustus', '08', regex=False
)

# Mengubah semua tanggal menjadi format tanggal
df['tanggal_ujian'] = pd.to_datetime(
    df['tanggal_ujian'],
    dayfirst=True,
    errors='coerce',
    format='mixed'
)

# Menampilkan format tanggal yang seragam
df['tanggal_ujian'] = df['tanggal_ujian'].dt.strftime('%Y-%m-%d')

print(df['tanggal_ujian'].head(10))

0    2026-08-06
1    2026-08-12
2    2026-08-15
3    2026-08-10
4    2026-08-06
5    2026-08-08
6    2026-08-19
7    2026-08-11
8    2026-08-20
9    2026-08-03
Name: tanggal_ujian, dtype: object


Cek Hasil Cleaning

In [43]:
print("Jumlah baris setelah cleaning:", len(df))
print("\nMissing value setelah cleaning:")
print(df.isnull().sum())

print("\nJumlah duplikat setelah cleaning:")
print(df.duplicated().sum())

print("\nTipe data setelah cleaning:")
print(df.dtypes)

print("\n5 data pertama setelah cleaning:")
print(df.head())

Jumlah baris setelah cleaning: 75

Missing value setelah cleaning:
id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             0
guru_pengampu     0
dtype: int64

Jumlah duplikat setelah cleaning:
0

Tipe data setelah cleaning:
id_siswa           object
nama               object
kelas              object
mata_pelajaran     object
jenis_ujian        object
tanggal_ujian      object
nilai             Float64
guru_pengampu      object
dtype: object

5 data pertama setelah cleaning:
  id_siswa           nama     kelas   mata_pelajaran jenis_ujian  \
0  SIS0004  Joko Prasetyo  XI RPL 2              KKA         uas   
1  SIS0020      Eka Putri  XI RPL 1              PKK         uts   
2  SIS0015  Nanda Pratama  XI RPL 3  Pemrograman Web         UAS   
3  SIS0046  Fajar Nugroho  XI RPL 2       Matematika          UH   
4  SIS0011    Ayu Lestari  XI RPL 2  Pemrograman Web          UH   

  tanggal_ujian  nilai guru_p

DATA MANIPULATION / ANALISIS

Analisis Pertanyaan No.1
Berapa jumlah data siswa dalam dataset?

In [44]:
print("Jumlah data siswa dalam dataset:", len(df))

Jumlah data siswa dalam dataset: 75


Analisis Pertanyaan No.2 Mata pelajaran mana yang memiliki rata-rata nilai paling tinggi dan paling rendah?

In [45]:
rata_nilai = df.groupby(
    'mata_pelajaran'
)['nilai'].mean()

print("Rata-rata nilai setiap mata pelajaran:")
print(rata_nilai.round(2))

Rata-rata nilai setiap mata pelajaran:
mata_pelajaran
Bahasa Inggris     66.76
Basis Data         59.87
KKA                64.25
Matematika         65.94
PKK                63.31
Pemrograman Web    66.91
Name: nilai, dtype: Float64


In [46]:
tertinggi = rata_nilai.idxmax()
nilai_tertinggi = rata_nilai.max()

print(
    "Rata-rata nilai tertinggi:",
    tertinggi,
    "=",
    round(nilai_tertinggi, 2)
)

Rata-rata nilai tertinggi: Pemrograman Web = 66.91


In [47]:
terendah = rata_nilai.idxmin()
nilai_terendah = rata_nilai.min()

print(
    "Rata-rata nilai terendah:",
    terendah,
    "=",
    round(nilai_terendah, 2)
)

Rata-rata nilai terendah: Basis Data = 59.87


Analisis Pertanyaan No3. Berapa data siswa yang kosong?

In [48]:
jumlah_data_kosong = df_awal.isnull().any(axis=1).sum()
print(
    "Jumlah data siswa yang memiliki data kosong:",
    jumlah_data_kosong
)

Jumlah data siswa yang memiliki data kosong: 8
